# GDFM Network Setup and Backbone Walkthrough

This notebook is about **model setup**, not training, sampling, or GDFM loss logic.

It is meant to answer four concrete questions:
- What does the canonical `gdfm.build_network(...)` API expect?
- Which concrete backbone do we get when we ask for `"mlp"`, `"unet"`, or `"jit"`?
- How large are these networks in parameter count, and where do those parameters live?
- Do the forward contracts actually work on vectors and images, with the expected shape changes?

What this notebook deliberately does **not** try to show:
- sample quality
- memorization behavior
- prediction-space or loss-space effects
- training dynamics

The outputs below are from **untrained** networks. The point is structural sanity checking.

## Roadmap

1. Build canonical nested configs for the three backbones.
2. Show how the factory normalizes those configs, including legacy aliases.
3. Instantiate the MLP, Bonnaire-style UNet, and vendored JiT wrapper.
4. Inspect parameter counts, grouped parameter budgets, and layer-wise shape traces.
5. Run one real forward pass for vectors, grayscale images, and RGB images.

In [ ]:
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

from pathlib import Path

import sys

WORKDIR = Path.cwd()
REPO_ROOT = WORKDIR if (WORKDIR / 'gdfm').exists() else WORKDIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from ml_collections import ConfigDict

from gdfm import build_network
from gdfm.model_setup import _normalize_network_config
from gdfm.notebook_utils import (
    backbone_row,
    make_grayscale_demo,
    make_rgb_demo,
    make_time_batch,
    make_vector_demo,
    parameter_groups,
    parameter_overview,
    parameter_table,
    plot_image_io,
    plot_vector_io,
    run_with_trace,
    set_seed,
    stats_table,
)

set_seed(7)
torch.set_grad_enabled(False)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.facecolor'] = 'white'

DEVICE = torch.device('cpu')
print('Using device:', DEVICE)
print('Notebook file:', Path('gdfm/model_setup_networks_walkthrough.ipynb').resolve())

## 1. Canonical model-setup surface

The long-term public API is the nested config surface:
- `cfg.network.arch`
- `cfg.network.*`
- `cfg.data.*`

We keep compatibility aliases during transition, but the notebook starts from the canonical nested shape.

In [ ]:
def make_mlp_config():
    cfg = ConfigDict()
    cfg.network = ConfigDict()
    cfg.network.arch = 'mlp'
    cfg.network.hidden_dim = 128
    cfg.network.n_layers = 3
    cfg.network.time_emb_dim = 64

    cfg.data = ConfigDict()
    cfg.data.data_dim = 32
    return cfg


def make_unet_config():
    cfg = ConfigDict()
    cfg.network = ConfigDict()
    cfg.network.arch = 'unet'
    cfg.network.base_channels = 64
    cfg.network.channel_mult = (1, 2, 4, 4)
    cfg.network.attn_resolutions = (8,)
    cfg.network.num_res_blocks = 2
    cfg.network.dropout = 0.0
    cfg.network.time_emb_dim = 256

    cfg.data = ConfigDict()
    cfg.data.image_size = 32
    cfg.data.image_channels = 1
    return cfg


def make_jit_config():
    cfg = ConfigDict()
    cfg.network = ConfigDict()
    cfg.network.arch = 'jit'
    cfg.network.patch_size = 16
    cfg.network.hidden_dim = 768
    cfg.network.depth = 12
    cfg.network.heads = 12
    cfg.network.bottleneck_dim = 128
    cfg.network.proj_dropout = 0.0
    cfg.network.attn_dropout = 0.0
    cfg.network.in_context_len = 32
    cfg.network.in_context_start = 4

    cfg.data = ConfigDict()
    cfg.data.image_size = 64
    cfg.data.image_channels = 3
    return cfg


mlp_cfg = make_mlp_config()
unet_cfg = make_unet_config()
jit_cfg = make_jit_config()

canonical_cfgs = {
    'mlp': mlp_cfg,
    'unet': unet_cfg,
    'jit': jit_cfg,
}

legacy_unet_cfg = {
    'network': 'unet',
    'model': {
        'in_channels': 1,
        'base_channels': 64,
        'channel_mults': [1, 2, 4, 4],
        'attention_resolutions': [8],
        'num_res_blocks': 2,
        'dropout': 0.0,
        'time_embed_dim': 256,
    },
    'data': {
        'resolution': 32,
    },
}

legacy_jit_cfg = {
    'network': 'jit',
    'model': {
        'in_channels': 3,
        'patch_size': 16,
        'hidden_dim': 768,
        'depth': 12,
        'num_heads': 12,
        'bottleneck_dim': 128,
        'dropout': 0.0,
    },
    'data': {
        'resolution': 64,
    },
}

In [ ]:
normalized_specs = pd.DataFrame(
    {
        name: pd.Series(_normalize_network_config(cfg))
        for name, cfg in canonical_cfgs.items()
    }
).T

print('Canonical specs after normalization:')
display(normalized_specs)

alias_translation = pd.DataFrame(
    {
        'canonical_unet': pd.Series(_normalize_network_config(unet_cfg)),
        'legacy_unet_aliases': pd.Series(_normalize_network_config(legacy_unet_cfg)),
        'canonical_jit': pd.Series(_normalize_network_config(jit_cfg)),
        'legacy_jit_aliases': pd.Series(_normalize_network_config(legacy_jit_cfg)),
    }
).fillna('')

print('The factory collapses legacy aliases onto the same public spec:')
display(alias_translation)

## 2. Instantiate the three backbones

This is the first sanity check: if we ask for a backbone, do we get the intended wrapper, and what concrete implementation does it contain internally?

In [ ]:
models = {
    name: build_network(cfg).to(DEVICE)
    for name, cfg in canonical_cfgs.items()
}

inputs = {
    'mlp': (
        make_vector_demo(mlp_cfg.data.data_dim, batch_size=1).to(DEVICE),
        make_time_batch(1, value=0.35).to(DEVICE),
    ),
    'unet': (
        make_grayscale_demo(unet_cfg.data.image_size, batch_size=1).to(DEVICE),
        make_time_batch(1, value=0.35).to(DEVICE),
    ),
    'jit': (
        make_rgb_demo(jit_cfg.data.image_size, batch_size=1).to(DEVICE),
        make_time_batch(1, value=0.35).to(DEVICE),
    ),
}

outputs = {}
rows = []
for name, model in models.items():
    z, t = inputs[name]
    y = model(z, t)
    outputs[name] = y

    inner = getattr(model, 'model', None)
    rows.append(
        {
            'backbone': name,
            'wrapper_class': model.__class__.__name__,
            'wrapper_module': model.__class__.__module__,
            'inner_class': inner.__class__.__name__ if inner is not None else '-',
            'inner_module': inner.__class__.__module__ if inner is not None else '-',
            'input_shape': tuple(z.shape),
            'output_shape': tuple(y.shape),
        }
    )

factory_table = pd.DataFrame(rows)
display(factory_table)

## 3. MLP deep dive

The MLP is the cleanest backbone to inspect because it acts directly on flat vectors.

Here we show:
- the concrete module structure
- the total and grouped parameter counts
- the exact sequence of learnable layers hit in one forward pass
- a structured vector input and its output

In [ ]:
mlp = models['mlp']
mlp_z, mlp_t = inputs['mlp']
mlp_y, mlp_trace = run_with_trace(mlp, mlp_z, mlp_t)

print(mlp)
display(parameter_overview(mlp))
display(parameter_groups(mlp, depth=1))
display(parameter_table(mlp))
display(stats_table([('input', mlp_z), ('output', mlp_y)]))
display(mlp_trace)

plot_vector_io(mlp_z, mlp_y);

## 4. Bonnaire-style UNet deep dive

For the UNet the main questions are different:
- does the wrapper really hold the Bonnaire-style implementation?
- how much of the parameter budget lives in the encoder/decoder trunk?
- do the spatial sizes and channel counts behave as expected across the forward pass?

The image shown below is a synthetic grayscale pattern. The output is still untrained, so we care about **contract sanity**, not semantic quality.

In [ ]:
unet = models['unet']
unet_z, unet_t = inputs['unet']
unet_y, unet_trace = run_with_trace(unet, unet_z, unet_t)

print(unet)
display(parameter_overview(unet))
display(parameter_groups(unet, depth=2).head(20))
display(parameter_table(unet).head(20))
display(stats_table([('input', unet_z), ('output', unet_y)]))
display(unet_trace)

plot_image_io(unet_z, unet_y);

## 5. JiT deep dive

The JiT wrapper is the most important setup sanity check because it is not a home-grown scaffold anymore: it wraps the vendored Li & He implementation and hides the dummy unconditional label handling inside the wrapper.

Besides the usual parameter and trace tables, we also inspect a few patch-level quantities:
- `patch_size`
- `patch_dim`
- `seq_length`
- `bottleneck_ratio`

In [ ]:
jit = models['jit']
jit_z, jit_t = inputs['jit']
jit_y, jit_trace = run_with_trace(jit, jit_z, jit_t)

print(jit)
display(
    pd.DataFrame(
        [
            {
                'image_size': jit.image_size,
                'image_channels': jit.image_channels,
                'patch_size': jit.patch_size,
                'patch_dim': jit.patch_dim,
                'seq_length': jit.seq_length,
                'hidden_dim': jit.hidden_dim,
                'depth': jit.depth,
                'heads': jit.heads,
                'bottleneck_dim': jit.bottleneck_dim,
                'bottleneck_ratio': round(jit.bottleneck_ratio, 3),
            }
        ]
    )
)
display(parameter_overview(jit))
display(parameter_groups(jit, depth=2).head(25))
display(parameter_table(jit).head(25))
display(stats_table([('input', jit_z), ('output', jit_y)]))
print(f'JiT trace length: {len(jit_trace)} rows')
display(jit_trace.head(60))

plot_image_io(jit_z, jit_y);

## 6. Cross-backbone summary

At this point we can compare the three backbones from the same factory entrypoint.

In [ ]:
comparison = pd.DataFrame(
    [
        backbone_row('mlp', mlp, mlp_z, mlp_y),
        backbone_row('unet', unet, unet_z, unet_y),
        backbone_row('jit', jit, jit_z, jit_y),
    ]
).sort_values('params')

display(comparison)

In [ ]:
assert tuple(mlp_y.shape) == tuple(mlp_z.shape)
assert tuple(unet_y.shape) == tuple(unet_z.shape)
assert tuple(jit_y.shape) == tuple(jit_z.shape)
assert factory_table.loc[factory_table.backbone == 'mlp', 'wrapper_class'].item() == 'TimeCondMLP'
assert factory_table.loc[factory_table.backbone == 'unet', 'wrapper_class'].item() == 'BonnaireUNet'
assert factory_table.loc[factory_table.backbone == 'jit', 'wrapper_class'].item() == 'JiT'

'All model-setup sanity checks passed.'

## Takeaway

This notebook should leave us with a concrete mental model of the current `gdfm` setup layer:
- the **factory surface** is the nested `cfg.network` / `cfg.data` API
- the **MLP** is the canonical flat-vector backbone for synthetic work
- the **UNet** wrapper really points at the Bonnaire-style image backbone already used elsewhere in the repo
- the **JiT** wrapper really points at the vendored Li & He implementation, with unconditional dummy-label handling hidden inside the wrapper
- all three backbones satisfy their expected input/output contracts on real forward passes

From here, the next refactors can focus on experiment wiring or GDFM integration without having to guess whether model setup itself is behaving.